[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-14-multilingual.ipynb)


# Lecture 14: one model, both directions

Until now each of your models has translated one way: A9 from English into your language and,
if you trained one in A13, a reverse model back. A multilingual model does both, and more, with
one set of weights. The trick is small: put a **tag** at the start of each source sentence that
says which language to produce, and train on both directions' pairs mixed together. That is
Assignment 14: from the same A9 split, pieces and configuration, build one model that translates
English into your language and back, and ask whether it matches two one-way models.

By the end you will be able to:

1. tag each pair with the language to produce and intermingle both directions into one training
   set, keeping both directions' test sentences out of it;
2. explain how a tag tells one model which way to translate, and why tagging lets a multilingual
   model translate between languages it never saw paired (zero-shot);
3. compare one model that translates both ways with two one-way models, with decoding pinned and
   the difference tested.

These serve Lecture 14's outcome *Explain tagging and why zero-shot works at all; build a
bidirectional system from their A9 one and score both directions*.

You write the two steps that are the method, tagging and intermingling, a hypothesis and a
prediction; checks tell you when each is right, and the notebook runs the rest.

**What you need.** Your A9 files in `MyDrive/CS479/assignment8`, as Lecture 9's Part B left
them: the split, `spm.model`, `a9-settings.json` and `best-a9/`. For the comparison into English,
what A13 left there: your reverse model, `best-a13-reverse/`, or, if NLLB back-translated for
you, NLLB again; the notebook finds which. Without either, you compare English to your language
only, and say so in the write-up.

**This is long.** One training, on twice A9's pairs (each pair once each way) for as many epochs
as A9, so about twice your A9 training. Use an A100, L4 or G4. Every step saves to Drive and can
be picked up in a new session: run the two setup cells, then the step you were on. **When you
finish, disconnect the GPU**: Runtime → Disconnect and delete runtime. Not while the training or
a decoding cell is running; the last cell says why.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 14 (*Multilingual NMT and "Zero-shot" NMT*). A head start on **A14**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Setup: the GPU, Drive, your A9 files, the configuration the training uses, and the helpers
# the steps below share. Run it first in every session.
import json
import random

import pandas as pd
import torch
import torchlingo
from torchlingo.checkpoint import load_checkpoint
from torchlingo.colab import course_dir, setup
from torchlingo.config import Config
from torchlingo.data_processing import create_dataloaders
from torchlingo.data_processing.vocab import SentencePieceVocab
from torchlingo.inference import translate_lines_to_file, translate_to_file
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
# Stops, naming each missing file, if A9 has not left them in MyDrive/CS479/assignment8.
A8_DIR = course_dir(
    env.drive_dir, "assignment8",
    needs=["train.tsv", "val.tsv", "test.tsv", "spm.model", "a9-settings.json",
           "best-a9/model_best.pt"],
    hint="Lecture 9's Part B writes them; finish A9 first.",
)

settings = json.loads((A8_DIR / "a9-settings.json").read_text(encoding="utf-8"))
NO_SPACES = settings["no_spaces"]
MAX_DECODE = settings["max_decode_length"]
TOKENIZE = "flores200" if NO_SPACES else "13a"     # the BLEU A9 was scored with
SPM = str(A8_DIR / "spm.model")
vocab = SentencePieceVocab(SPM)         # A9's pieces, trained on both languages

# The training runs as many epochs as A9 did, read from A9's checkpoint on Drive, so each
# direction sees its pairs as often as a one-way model does.
a9_run = TrainingCheckpointer(experiment_name="assignment-9", verbose=False)
if a9_run.has_checkpoint():
    a9_state = torch.load(a9_run.path_for("latest"), map_location="cpu", weights_only=False)["state"]
    EPOCHS = len(a9_state["train_losses"])
else:
    EPOCHS = None   # <- only if no A9 checkpoint is found: the epochs your A9 notebook trained for
assert EPOCHS, (
    f"No A9 checkpoint at {a9_run.path_for('latest')}. Set EPOCHS on the line above to the "
    "number of epochs your A9 notebook trained for, and run this cell again."
)
SEED = 479
config = Config(
    d_model=512,
    n_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    d_ff=2048,
    batch_size=64,
    num_workers=0,
    max_decode_length=MAX_DECODE,
)
NOTES = A8_DIR / "a14-notes.json"   # what each step saves for the write-up


def load_notes():
    """Everything the steps have saved so far, so a dropped session loses none of it."""
    return json.loads(NOTES.read_text(encoding="utf-8")) if NOTES.exists() else {}


def save_note(**items):
    """Add items to a14-notes.json on Drive."""
    NOTES.write_text(json.dumps({**load_notes(), **items}, indent=2, ensure_ascii=False),
                     encoding="utf-8")


def read_tsv(name):
    return pd.read_csv(A8_DIR / name, sep="\t", dtype=str, keep_default_na=False)


def new_model():
    torch.manual_seed(SEED)
    return SimpleTransformer(src_vocab_size=len(vocab), tgt_vocab_size=len(vocab), config=config).to(device)


def train(train_tsv, val_tsv, run_name, best_dir):
    """A9's training call on new files; resumes from its checkpoint if the session dropped."""
    random.seed(SEED)
    train_loader, val_loader, _, _ = create_dataloaders(
        A8_DIR / train_tsv, A8_DIR / val_tsv, use_bucketing=True, num_workers=0, config=config,
        use_sentencepiece=True, sp_model_path=SPM, sp_tgt_model_path=SPM,
    )
    model = new_model()
    result = train_model(model, train_loader, val_loader=val_loader, num_epochs=EPOCHS,
                         device=device, config=config, gradient_clip=1.0, save_dir=A8_DIR / best_dir,
                         log_every=50, checkpointer=TrainingCheckpointer(experiment_name=run_name))
    print(f"Epochs: {len(result.train_losses)}   final val loss {result.val_losses[-1]:.4f}")


def load_best(best_dir):
    model = new_model()
    model.load_state_dict(load_checkpoint(A8_DIR / best_dir / "model_best.pt", map_location=device)["model_state_dict"])
    return model.eval()


# The one-way model into English, for the comparison: what A13 left, your reverse model or NLLB.
a13_notes_file = A8_DIR / "a13-notes.json"
a13_notes = json.loads(a13_notes_file.read_text(encoding="utf-8")) if a13_notes_file.exists() else {}
if (A8_DIR / "best-a13-reverse" / "model_best.pt").exists():
    INTO_ENGLISH = "A13 reverse"
elif a13_notes.get("back_translator") == "nllb":
    INTO_ENGLISH = "NLLB-200"
else:
    INTO_ENGLISH = None
_nllb = {}


def into_english(sentences, path, decoder):
    """Translate sentences in your language into English with A13's one-way model."""
    if INTO_ENGLISH == "NLLB-200":
        from torchlingo_harness.nllb import load_nllb   # an accessory shipped with TorchLingo

        if "model" not in _nllb:
            _nllb["model"] = load_nllb(a13_notes["nllb_code"], "eng_Latn", device=device)
        nllb = _nllb["model"]
        options = {"decode_strategy": decoder, "beam_size": 5}
        return translate_lines_to_file(lambda chunk, first: nllb(chunk, **options), sentences,
                                       path, nllb.settings(**options), verbose=False)
    return translate_to_file(load_best("best-a13-reverse"), sentences, vocab, vocab, path,
                             decode_strategy=decoder, beam_size=5, device=device, config=config,
                             verbose=False)


train_df, val_df, test_df = read_tsv("train.tsv"), read_tsv("val.tsv"), read_tsv("test.tsv")
print(f"Your split: {len(train_df):,} train, {len(val_df):,} validation, {len(test_df):,} test pairs.")
print(f"A9 trained for {EPOCHS} epochs; the A14 training runs the same.")
print(f"Into English, A14 is compared with: {INTO_ENGLISH or 'nothing (no reverse model or NLLB from A13)'}")

## Step 1: the tags

A tag is ordinary text at the start of the source sentence: `<2x>` for "translate into your
language" and `<2en>` for "translate into English". It is the only thing that tells the model
which way to go, since the source alone does not say which language the output should be in.

Your A9 pieces were trained without these tags, so SentencePiece splits each into a few pieces,
and a character it never saw, such as `<`, becomes `<unk>`. That is fine as long as the two tags
come out different, because the difference is what the model learns to read. (Large systems
register each tag as a single piece; keeping A9's pieces here keeps the vocabulary out of the
comparison.) The cell shows each tag's pieces and stops if the two are the same.

In [ ]:
TO_X, TO_EN = "<2x>", "<2en>"   # the language to produce: yours, or English

pieces = {t: [vocab.idx_to_token(i) for i in vocab.encode(t, add_special_tokens=False)]
          for t in (TO_X, TO_EN)}
for t, p in pieces.items():
    print(f"{t:7} becomes {len(p)} pieces: {p}")
assert pieces[TO_X] != pieces[TO_EN], (
    "The two tags become the same pieces, so the model could not tell them apart. Change a "
    "letter in one of them, and run this cell again.")
save_note(tags=[TO_X, TO_EN], tag_pieces=pieces)

## Step 2: tag, intermingle, and keep the test sentences out

**Yours to write: `tag` and `both_directions`.**

- `tag(frame, tag)` returns the same pairs with the tag and a space in front of every source
  sentence, leaving the table it was given unchanged.
- `both_directions(frame)` returns every pair twice: English to your language, tagged `TO_X`,
  and the same pair turned around, your sentence to its English, tagged `TO_EN`. Then it
  **intermingles** the two halves by shuffling them with `SEED`. Trained on one direction and
  then the other, a model drifts toward whichever came last; mixed, every batch holds both.

The cell checks both functions on a known case, then builds `train-a14.tsv` and `val-a14.tsv`
from A9's training and validation sets, and one test set per direction from A9's test set,
`test-a14-to-x.tsv` and `test-a14-to-en.tsv`. Every test sentence is now a source in one
direction and a reference in the other, so the contamination check runs on both languages: no
test sentence, English or yours, may be anywhere in training.

In [ ]:
from torchlingo.diagnostics import check_contamination


def tag(frame, tag):
    """Return the same pairs with tag, then a space, at the start of every source sentence.

    TODO: write it, in a line or two. Keep the two columns named src and tgt, in that order.
    """
    raise NotImplementedError("write tag, then run this cell again")


def both_directions(frame):
    """Return every pair once each way, tagged with the language to produce, intermingled.

    TODO: write it, as the markdown above describes, using tag.
    """
    raise NotImplementedError("write both_directions, then run this cell again")


# A case whose answer you know: two pairs.
toy = pd.DataFrame({"src": ["good morning", "thank you"], "tgt": ["buenos días", "gracias"]})
tagged = tag(toy, TO_X)
expected = [[f"{TO_X} good morning", "buenos días"], [f"{TO_X} thank you", "gracias"]]
assert list(tagged.columns) == ["src", "tgt"], "keep the columns named src and tgt, in that order"
assert tagged.values.tolist() == expected, (
    f"tag is not right yet: for this case it should return {expected}, and yours returned "
    f"{tagged.values.tolist()}")
assert toy.src.iloc[0] == "good morning", "tag changed the table it was given; work on a copy"

both = both_directions(toy)
expected = sorted([(f"{TO_X} good morning", "buenos días"), (f"{TO_X} thank you", "gracias"),
                   (f"{TO_EN} buenos días", "good morning"), (f"{TO_EN} gracias", "thank you")])
assert list(both.columns) == ["src", "tgt"], "keep the columns named src and tgt, in that order"
assert sorted(map(tuple, both.values.tolist())) == expected, (
    f"both_directions is not right yet: for this case it should return these four pairs, in "
    f"any order, {expected}, and yours returned {both.values.tolist()}")
assert both.equals(both_directions(toy)), "the same SEED must give the same order every time"

# Your split, both ways. The first half must hold both directions, or they are not intermingled.
combined = both_directions(train_df[["src", "tgt"]])
share = combined.src.iloc[:len(combined) // 2].str.startswith(TO_X).mean()
assert 0.2 < share < 0.8, (
    f"{share:.0%} of the first half of the training set goes into your language: the two "
    "directions are not intermingled. Shuffle the pairs after putting them together.")
tests = {
    "to-x": tag(test_df[["src", "tgt"]], TO_X),
    "to-en": tag(test_df.rename(columns={"src": "tgt", "tgt": "src"})[["src", "tgt"]], TO_EN),
}

# No test sentence, in either language, may be anywhere in training, source or target.
in_training = list(combined.src.str.split(" ", n=1).str[1]) + list(combined.tgt)
for language, sentences in (("English", test_df.src), ("your language", test_df.tgt)):
    result = check_contamination(sentences, in_training, name=f"test, {language}")
    print(result)
    assert result, "a test sentence is in the training set; it would put the answer into training"

combined.to_csv(A8_DIR / "train-a14.tsv", sep="\t", index=False)
both_directions(val_df[["src", "tgt"]]).to_csv(A8_DIR / "val-a14.tsv", sep="\t", index=False)
for name, frame in tests.items():
    frame.to_csv(A8_DIR / f"test-a14-{name}.tsv", sep="\t", index=False)
save_note(training_pairs=len(combined))
print(f"Written: train-a14.tsv, {len(combined):,} pairs from A9's {len(train_df):,}; val-a14.tsv; "
      "test-a14-to-x.tsv and test-a14-to-en.tsv.")
print(f"For example:\n  {combined.src.iloc[0]}  ->  {combined.tgt.iloc[0]}")

## Step 3: state a hypothesis and a prediction, then train

Before the model trains, **state your hypothesis**, the Final Project's Path A in small: *"If one
model learns both directions, then its English-to-X BLEU on my A8 test set will ..."*, finished
with what you expect against A9 and why. Then **a prediction**: roughly how far each direction
will land from its one-way model, A9 into your language and, into English, your A13 reverse
model or NLLB. One model now shares its weights between two jobs with the same training budget
per direction: say whether you expect that to cost or to help. The notebook stops until you
write both, and saves them to Drive for the write-up.

The cell after that is the long training, with A9's configuration, pieces and epochs, so the
difference from A9 is the second direction and the tags.

In [ ]:
HYPOTHESIS = ""   # <- "If one model learns both directions, then its English-to-X BLEU ... will ..."
PREDICTION = ""   # <- how far each direction will land from its one-way model, and why

assert HYPOTHESIS.strip(), "Write your hypothesis in HYPOTHESIS, then run this cell again."
assert HYPOTHESIS.strip().lower().startswith("if") and " then " in HYPOTHESIS.lower(), (
    'Write the hypothesis as "If ..., then ...": what you change, and what you expect it to do.')
assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."
save_note(hypothesis=HYPOTHESIS.strip(), prediction=PREDICTION.strip())
print("Hypothesis and prediction saved to a14-notes.json, for the write-up.")

In [ ]:
# The long training: one model, both directions. If the session drops, run this again: an
# unfinished training resumes, a finished one is skipped.
train("train-a14.tsv", "val-a14.tsv", "assignment-14", "best-a14")

## Step 4: score each direction against its one-way model

The A14 model translates both test sets: into your language, compared with A9, and into
English, compared with A13's reverse model or NLLB. Every system decodes **greedily and with
beam search (k = 5)**, and a comparison is only ever between two systems decoded the same way,
as in A13. Into your language the BLEU is A9's; into English it is SacreBLEU's default; chrF
for both. **sacreBLEU's paired bootstrap** gives each score a 95% interval and a *p* for A14
against the one-way model, as in A13.

A9's outputs from A13 are reused if they are on Drive. The beam half is slow, one sentence at a
time; everything is saved to Drive as it goes.

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
from sacrebleu.significance import PairedTest

# Each direction: its references, its BLEU, and the one-way model it is compared with.
DIRECTIONS = {
    "to-x": {"name": "into your language", "refs": list(test_df.tgt), "tokenize": TOKENIZE,
             "one_way": "A9"},
    "to-en": {"name": "into English", "refs": list(test_df.src), "tokenize": "13a",
              "one_way": INTO_ENGLISH},
}

# Every system decodes its test set both ways, each to its own file on Drive. A decode that is
# interrupted resumes when you run the cell again. If you retrain, delete its .hyp files first.
hyps, hyp_files = {}, []
for decoder in ("greedy", "beam"):
    for direction in DIRECTIONS:
        path = A8_DIR / f"test-a14-{direction}-{decoder}.hyp"
        print(f"A14, {direction}, {decoder}: {path.name}")
        sources = list(read_tsv(f"test-a14-{direction}.tsv").src)
        hyps["A14", direction, decoder] = translate_to_file(
            load_best("best-a14"), sources, vocab, vocab, path, decode_strategy=decoder,
            beam_size=5, device=device, config=config, verbose=False)
        hyp_files.append(path.name)
    path = A8_DIR / f"test-a9-{decoder}.hyp"
    print(f"A9, to-x, {decoder}: {path.name}")
    hyps["A9", "to-x", decoder] = translate_to_file(
        load_best("best-a9"), list(test_df.src), vocab, vocab, path, decode_strategy=decoder,
        beam_size=5, device=device, config=config, verbose=False)
    hyp_files.append(path.name)
    if INTO_ENGLISH:
        path = A8_DIR / f"test-a13-reverse-{decoder}.hyp"
        print(f"{INTO_ENGLISH}, to-en, {decoder}: {path.name}")
        hyps[INTO_ENGLISH, "to-en", decoder] = into_english(list(test_df.tgt), path, decoder)
        hyp_files.append(path.name)


def bootstrap(direction, decoder):
    """BLEU and chrF for A14 and the one-way model it is compared with, each with a 95%
    interval, and A14's p against it, from sacreBLEU's paired bootstrap over 1,000 resamples."""
    d = DIRECTIONS[direction]
    names = [n for n in (d["one_way"], "A14") if n]
    metrics = {"BLEU": BLEU(tokenize=d["tokenize"]), "chrF": CHRF()}

    def number(x):
        return None if x is None else round(float(x), 4)

    if len(names) == 1:   # nothing to compare with: A14's scores alone
        h = hyps["A14", direction, decoder]
        b, c = (m.corpus_score(h, [d["refs"]]) for m in metrics.values())
        return {"A14": {"bleu": number(b.score), "bleu_ci": None, "bleu_p": None,
                        "chrf": number(c.score), "chrf_ci": None, "chrf_p": None}}
    _, result = PairedTest([(n, hyps[n, direction, decoder]) for n in names], metrics,
                           [d["refs"]], test_type="bs", n_samples=1000)()
    bleu, chrf = (result[key] for key in result if key != "System")
    scored = {n: {"bleu": number(b.score), "bleu_ci": number(b.ci), "bleu_p": number(b.p_value),
                  "chrf": number(c.score), "chrf_ci": number(c.ci), "chrf_p": number(c.p_value)}
              for n, b, c in zip(names, bleu, chrf)}
    # Identical outputs would come out at sacreBLEU's smallest p, 0.001, "real". No difference is 1.
    if hyps[names[0], direction, decoder] == hyps["A14", direction, decoder]:
        scored["A14"].update(bleu_p=1.0, chrf_p=1.0)
    return scored


def show(scores):
    """Print a direction's rows: system, BLEU and chrF with their intervals, and A14's p."""
    for system, s in scores.items():
        ci = "" if s["bleu_ci"] is None else f" +/- {s['bleu_ci']:.2f}"
        chrf_ci = "" if s["chrf_ci"] is None else f" +/- {s['chrf_ci']:.2f}"
        p = "" if s["bleu_p"] is None else f"   p = {s['bleu_p']:.3f} (BLEU), {s['chrf_p']:.3f} (chrF)"
        print(f"    {system:<12} BLEU {s['bleu']:6.2f}{ci:<9}  chrF {s['chrf']:6.2f}{chrf_ci:<9}{p}")


scores = {direction: {decoder: bootstrap(direction, decoder) for decoder in ("greedy", "beam")}
          for direction in DIRECTIONS}
save_note(scores=scores, hyp_files=hyp_files, into_english=INTO_ENGLISH)
for direction, d in DIRECTIONS.items():
    for decoder in ("greedy", "beam"):
        print(f"\n{d['name']}, {decoder}:")
        show(scores[direction][decoder])

## Step 5: what the tag does

The tag is the only thing that says which way to go, so give the model the wrong one: five of
your test set's English sentences tagged *into English*, and the same five sentences in your
language tagged *into your language*. Does it copy, paraphrase, translate anyway, or produce
something else? Whatever it does shows what it learned the tag means. The write-up asks about
it.

In [ ]:
model = load_best("best-a14")
picked = test_df.head(5)
wrong = {
    "English, tagged into English": [f"{TO_EN} {s}" for s in picked.src],
    "your language, tagged into your language": [f"{TO_X} {s}" for s in picked.tgt],
}
shown = {}
for i, (title, sources) in enumerate(wrong.items()):
    out = translate_to_file(model, sources, vocab, vocab, A8_DIR / f"wrong-tag-{i}.hyp",
                            decode_strategy="greedy", device=device, config=config,
                            overwrite=True, verbose=False)
    shown[title] = [[s.split(" ", 1)[1], o] for s, o in zip(sources, out)]
    print(f"=== {title} ===")
    for source, output in shown[title]:
        print(f"  in:  {source}\n  out: {output}")
save_note(wrong_tag=shown)

## Step 6: the write-up

The cell below prints the numbers in the order the write-up asks for and saves `a14-rerun.json`.
It reads everything from Drive, so in a new session it needs only the setup cell. Answer these in
your write-up:

1. **Your hypothesis against the result.** Quote your "If ..., then ..." and your prediction,
   then give BLEU and chrF, with their intervals, for each direction and its one-way model,
   greedy and beam. Did the paired bootstrap find each difference real (p below 0.05)?
2. **Into your language: did sharing the model with the other direction cost or help, against
   A9?** Each direction saw its pairs as often as A9 did; say what else changed for the model,
   and which of those changes you think the result reflects.
3. **Into English: how does the A14 model compare with your A13 reverse model?** If you compared
   with NLLB instead, say so, and say what the comparison can and cannot tell you, given how
   different the two models are. If you had neither, say what you would expect and why.
4. **How does one model know which way to translate?** Use Step 1's pieces and Step 5's
   wrong-tag outputs: what did your model learn the tag means, and what did it do when the tag
   asked for the language it was given?
5. **Why does zero-shot work at all?** A model trained on English with X and English with Y,
   tagged, can be asked to translate X into Y, a pair it never saw. Say what the model must share
   across languages for that to work, why it may still answer in the wrong language, and why your
   two-language model cannot show it.
6. **How would someone rerun this?** Paste the contents of `a14-rerun.json`.

**Upload** to your folder in the Assignment shared folder: `train-a14.tsv`, the two test sets
`test-a14-to-x.tsv` and `test-a14-to-en.tsv`, every `.hyp` file Step 4 wrote, `a14-rerun.json`,
the write-up, and **this notebook**: File → Download → Download .ipynb, after it has run, so every
cell's output is in it.

In [ ]:
# Reads everything from Drive, so after a dropped session it needs only the setup cell above.
import importlib.metadata

notes = load_notes()
names = {"to-x": "into your language", "to-en": "into English"}
which = "spBLEU (flores200)" if NO_SPACES else "SacreBLEU, default tokenizer"

print("=== For the A14 write-up ===")
print(f"Hypothesis, written before training: {notes['hypothesis']}")
print(f"Prediction, written before training: {notes['prediction']}\n")
print(f"Tags: {notes['tags'][0]} into your language, {notes['tags'][1]} into English; as pieces: "
      + ";  ".join(f"{t} {p}" for t, p in notes["tag_pieces"].items()))
print(f"Training pairs: {notes['training_pairs']:,}, A9's split once each way; {EPOCHS} epochs")
print(f"Into English, compared with: {notes['into_english'] or 'nothing (no A13 model)'}\n")

print(f"BLEU into your language is {which}; into English, SacreBLEU's default. Each score is "
      "followed by its 95% interval and, for A14, the paired bootstrap's p against the one-way "
      "model decoded the same way:")
for direction, by_decoder in notes["scores"].items():
    for decoder, rows in by_decoder.items():
        print(f"\n  {names[direction]}, {decoder}:")
        for system, s in rows.items():
            ci = "" if s["bleu_ci"] is None else f" +/- {s['bleu_ci']:.2f}"
            chrf_ci = "" if s["chrf_ci"] is None else f" +/- {s['chrf_ci']:.2f}"
            p = "" if s["bleu_p"] is None else f"   p = {s['bleu_p']:.3f} (BLEU), {s['chrf_p']:.3f} (chrF)"
            print(f"    {system:<12} BLEU {s['bleu']:6.2f}{ci:<9}  chrF {s['chrf']:6.2f}{chrf_ci:<9}{p}")

for title, rows in notes.get("wrong_tag", {}).items():
    print(f"\n=== The wrong tag: {title} ===")
    for source, output in rows:
        print(f"  in:  {source}\n  out: {output}")

# Everything someone needs to rerun A14 and get the same numbers.
rerun = {
    "tags": notes["tags"], "seed": SEED, "epochs": EPOCHS,
    "training": "A9's train.tsv once each way, tagged and shuffled with the seed",
    "configuration": "A9's: d_model 512, 8 heads, 6+6 layers, d_ff 2048, batch 64, A9's SentencePiece",
    "scoring": f"greedy, and beam 5 with alpha 0.6; max length {MAX_DECODE}; into your language "
               f"{which}, into English SacreBLEU default; chrF; paired bootstrap, 1,000 samples",
    "into_english_compared_with": notes["into_english"],
    "torchlingo": torchlingo.version_label(), "sacrebleu": importlib.metadata.version("sacrebleu"),
    "torch": torch.__version__,
}
(A8_DIR / "a14-rerun.json").write_text(json.dumps(rerun, indent=2, ensure_ascii=False) + "\n",
                                       encoding="utf-8")
print(f"\nSaved {A8_DIR / 'a14-rerun.json'}: paste it into the write-up (question 6).")

---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. **Not while the training or a decoding cell is running**: ending the session stops it (it resumes in a new session, but the time in between is lost). Everything A14 makes is on Drive, so disconnecting at the end loses nothing.